In [1]:
import os
import torch
# from transformers import AutoImageProcessor, UperNetForSemanticSegmentation
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [2]:
model_id = "facebook/mask2former-swin-large-ade-semantic"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
image_processor = AutoImageProcessor.from_pretrained(model_id)
image_segmentor = Mask2FormerForUniversalSegmentation.from_pretrained(model_id)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
C:\Users\Rohit\.conda\envs\transformers\lib\site-packages\transformers\image_processing_base.py:417: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


Load Data

In [ ]:
set_name = ''
img_folder = r""
locs_path = r""

In [5]:
loc_df = pd.read_csv(locs_path)
loc_df.head()

,uuid,source,orig_id,lat,lon,city,filename
0,13868c9e-403f-470f-82d2-cf984cc5c360,Mapillary,469091034374642,55.752541,37.623845,Moscow,13868c9e-403f-470f-82d2-cf984cc5c360.jpeg
1,dc6d37e7-3d36-498e-adc5-2d17e51b028b,Mapillary,893791088148073,55.764147,37.618920,Moscow,dc6d37e7-3d36-498e-adc5-2d17e51b028b.jpeg
2,d056b091-7fe9-4bec-aa92-bf27c3721af6,Mapillary,1176871832792035,55.762321,37.619977,Moscow,d056b091-7fe9-4bec-aa92-bf27c3721af6.jpeg
3,e2d69742-4833-4caa-8a2b-5f34fdfd8abf,Mapillary,229153325243992,55.753018,37.623222,Moscow,e2d69742-4833-4caa-8a2b-5f34fdfd8abf.jpeg
4,8372853e-c34d-41f4-a7bf-f7a4406852eb,Mapillary,969170933654386,55.761847,37.627047,Moscow,8372853e-c34d-41f4-a7bf-f7a4406852eb.jpeg


In [6]:
palette = np.asarray([[120, 120, 120], [180, 120, 120], [6, 230, 230], [80, 50, 50], [4, 200, 3], [120, 120, 80], [140, 140, 140], [204, 5, 255], [230, 230, 230], [4, 250, 7], [224, 5, 255], [235, 255, 7], [150, 5, 61], [120, 120, 70], [8, 255, 51], [255, 6, 82], [143, 255, 140], [204, 255, 4], [255, 51, 7], [204, 70, 3], [0, 102, 200], [61, 230, 250], [255, 6, 51], [11, 102, 255], [255, 7, 71], [255, 9, 224], [9, 7, 230], [220, 220, 220], [255, 9, 92], [112, 9, 255], [8, 255, 214], [7, 255, 224], [255, 184, 6], [10, 255, 71], [255, 41, 10], [7, 255, 255], [224, 255, 8], [102, 8, 255], [255, 61, 6], [255, 194, 7], [255, 122, 8], [0, 255, 20], [255, 8, 41], [255, 5, 153], [6, 51, 255], [235, 12, 255], [160, 150, 20], [0, 163, 255], [140, 140, 140], [250, 10, 15], [20, 255, 0], [31, 255, 0], [255, 31, 0], [255, 224, 0], [153, 255, 0], [0, 0, 255], [255, 71, 0], [0, 235, 255], [0, 173, 255], [31, 0, 255], [11, 200, 200], [255, 82, 0], [0, 255, 245], [0, 61, 255], [0, 255, 112], [0, 255, 133], [255, 0, 0], [255, 163, 0], [255, 102, 0], [194, 255, 0], [0, 143, 255], [51, 255, 0], [0, 82, 255], [0, 255, 41], [0, 255, 173], [10, 0, 255], [173, 255, 0], [0, 255, 153], [255, 92, 0], [255, 0, 255], [255, 0, 245], [255, 0, 102], [255, 173, 0], [255, 0, 20], [255, 184, 184], [0, 31, 255], [0, 255, 61], [0, 71, 255], [255, 0, 204], [0, 255, 194], [0, 255, 82], [0, 10, 255], [0, 112, 255], [51, 0, 255], [0, 194, 255], [0, 122, 255], [0, 255, 163], [255, 153, 0], [0, 255, 10], [255, 112, 0], [143, 255, 0], [82, 0, 255], [163, 255, 0], [255, 235, 0], [8, 184, 170], [133, 0, 255], [0, 255, 92], [184, 0, 255], [255, 0, 31], [0, 184, 255], [0, 214, 255], [255, 0, 112], [92, 255, 0], [0, 224, 255], [112, 224, 255], [70, 184, 160], [163, 0, 255], [153, 0, 255], [71, 255, 0], [255, 0, 163], [255, 204, 0], [255, 0, 143], [0, 255, 235], [133, 255, 0], [255, 0, 235], [245, 0, 255], [255, 0, 122], [255, 245, 0], [10, 190, 212], [214, 255, 0], [0, 204, 255], [20, 0, 255], [255, 255, 0], [0, 153, 255], [0, 41, 255], [0, 255, 204], [41, 0, 255], [41, 255, 0], [173, 0, 255], [0, 245, 255], [71, 0, 255], [122, 0, 255], [0, 255, 184], [0, 92, 255], [184, 255, 0], [0, 133, 255], [255, 214, 0], [25, 194, 194], [102, 255, 0], [92, 0, 255]])
labels = ['wall', 'building', 'sky', 'floor', 'tree', 'ceiling', 'road', 'bed', 'window', 'grass', 'cabinet', 'sidewalk', 'person', 'earth', 'door', 'table', 'mountain', 'plant', 'curtain', 'chair', 'car', 'water', 'painting', 'sofa', 'shelf', 'house', 'sea', 'mirror', 'rug', 'field', 'armchair', 'seat', 'fence', 'desk', 'rock', 'wardrobe', 'lamp', 'bathtub', 'railing', 'cushion', 'base', 'box', 'column', 'signboard', 'dresser', 'counter', 'sand', 'sink', 'skyscraper', 'fireplace', 'refrigerator', 'grandstand', 'path', 'stairs', 'runway', 'case', 'pooltable', 'pillow', 'screen', 'stairway', 'river', 'bridge', 'bookcase', 'blind', 'coffeetable', 'toilet', 'flower', 'book', 'hill', 'bench', 'countertop', 'stove', 'palmtree', 'kitchen', 'computer', 'swivelchair', 'boat', 'bar', 'arcade', 'hut', 'bus', 'towel', 'light', 'truck', 'tower', 'chandelier', 'awning', 'streetlight', 'booth', 'television', 'airplane', 'dirttrack', 'apparel', 'pole', 'land', 'balustrade', 'escalator', 'ottoman', 'bottle', 'sideboard', 'poster', 'stage', 'van', 'ship', 'fountain', 'conveyerbelt', 'canopy', 'washer', 'toy', 'pool', 'stool', 'barrel', 'basket', 'waterfall', 'tent', 'bag', 'motorbike', 'cradle', 'oven', 'ball', 'food', 'step', 'tank', 'brandname', 'microwave', 'pot', 'animal', 'bicycle', 'lake', 'dishwasher', 'screen.1', 'blanket', 'sculpture', 'hood', 'sconce', 'vase', 'trafficlight', 'tray', 'trashcan', 'fan', 'pier', 'crtscreen', 'plate', 'monitor', 'bulletinboard', 'shower', 'radiator', 'glass', 'clock', 'flag']

In [7]:
print(len(labels))
print(len(palette))

150
150


In [8]:
df_filenames = list(loc_df['filename'])
len(df_filenames)

29999

In [9]:
start_index = 0
end_index = len(df_filenames)
batch_size = 16

feats = []

image_segmentor = image_segmentor.to(device)
image_segmentor.eval()

palette_arr = np.array(palette, dtype=np.uint8)

save_seg_dir = "seg_outputs"
os.makedirs(save_seg_dir, exist_ok=True)

for batch_start in range(start_index, end_index, batch_size):
    batch_end = min(batch_start + batch_size, end_index)

    images = []
    valid_rows = []
    filenames = []

    for i in range(batch_start, batch_end):
        filename = df_filenames[i]
        img_path = os.path.join(img_folder, filename)

        try:
            image = Image.open(img_path).convert("RGB")
            images.append(image)
            valid_rows.append(i)
            filenames.append(filename)
        except Exception as e:
            print(f"Failed to load pano index {i} | File {filename}: {e}")

    if not images:
        continue

    print(f"Processing batch {batch_start}:{batch_end-1} ({len(images)} images)")

    inputs = image_processor(images=images, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    with torch.no_grad():
        outputs = image_segmentor(pixel_values)

    seg_list = image_processor.post_process_semantic_segmentation(
        outputs,
        target_sizes=[img.size[::-1] for img in images]
    )

    for k, seg in enumerate(seg_list):
        i = valid_rows[k]
        filename = filenames[k]

        seg = seg.cpu().numpy()

        # save color segmentation
        seg_rgb = palette_arr[seg]
        Image.fromarray(seg_rgb).save(
            os.path.join(save_seg_dir, filename)
        )

        # class proportions
        total_pixels = seg.size
        counts = np.bincount(seg.ravel(), minlength=len(labels)) / total_pixels
        feats.append(counts)
feats = np.vstack(feats)

Processing batch 0:15 (16 images)
Processing batch 16:31 (16 images)
Processing batch 32:47 (16 images)
Processing batch 48:63 (16 images)
Processing batch 64:79 (16 images)
Processing batch 80:95 (16 images)
Processing batch 96:111 (16 images)
Processing batch 112:127 (16 images)
Processing batch 128:143 (16 images)
Processing batch 144:159 (16 images)
Processing batch 160:175 (16 images)
Processing batch 176:191 (16 images)
Processing batch 192:207 (16 images)
Processing batch 208:223 (16 images)
Processing batch 224:239 (16 images)
Processing batch 240:255 (16 images)
Processing batch 256:271 (16 images)
Processing batch 272:287 (16 images)
Processing batch 288:303 (16 images)
Processing batch 304:319 (16 images)
Processing batch 320:335 (16 images)
Processing batch 336:351 (16 images)
Processing batch 352:367 (16 images)
Processing batch 368:383 (16 images)
Processing batch 384:399 (16 images)
Processing batch 400:415 (16 images)
Processing batch 416:431 (16 images)
Processing batc

In [10]:
np.save(f'mask2former_{set_name}.npy', feats)

In [11]:
feats.shape

(29999, 150)